# CMT — ARCADE Training Pipeline

**Phase 1:** Pretrain ResNet50 backbone on ARCADE (Places365 style)  
**Phase 2:** Train CMT inpainting model using pretrained backbone

---
⚠️ **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────
!git clone https://github.com/C0d3Crush/CMT.git
%cd CMT

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────
!pip install -q torch torchvision tqdm opencv-python-headless

In [ ]:
# ── 4. Upload & unzip ARCADE dataset ─────────────────────────────────────
# Upload arcade.zip via the left sidebar → Files → Upload
# then run this cell
!unzip -q -o /content/arcade.zip -d /content/CMT/
!find arcade -name '*.json' | head -5

---
## Phase 1 — Backbone Pretraining

In [ ]:
# ── 5. Convert COCO → classification layout ───────────────────────────────
!python coco_to_classification.py \
    --annotations arcade/syntax/train/annotations/train.json \
    --images      arcade/syntax/train/images \
    --output      data/train

!python coco_to_classification.py \
    --annotations arcade/syntax/val/annotations/val.json \
    --images      arcade/syntax/val/images \
    --output      data/val

In [ ]:
# ── 6. Remove empty class folders ────────────────────────────────────────
!find data/train -type d -empty -delete
!find data/val   -type d -empty -delete
!echo "Classes in train:" && ls data/train | wc -l
!echo "Classes in val:"   && ls data/val   | wc -l

In [ ]:
# ── 7. Pretrain backbone ──────────────────────────────────────────────────
!python train_placesCNN.py data/ \
    --arch        resnet50 \
    --num_classes 26 \
    --epochs      30 \
    --batch-size  128 \
    --lr          0.01 \
    --workers     2

In [ ]:
# ── 8. Link pretrained backbone as place2.pth ─────────────────────────────
import glob, os
best = glob.glob('checkpoints/*_best.pth.tar')
if best:
    if os.path.exists('place2.pth') or os.path.islink('place2.pth'):
        os.remove('place2.pth')
    os.symlink(os.path.abspath(best[0]), 'place2.pth')
    print(f'place2.pth → {best[0]}')
else:
    print('ERROR: no checkpoint found')

---
## Phase 2 — CMT Inpainting Training

In [ ]:
# ── 9. Train CMT ──────────────────────────────────────────────────────────
!python train.py \
    --train_img  arcade/syntax/train/images \
    --train_ann  arcade/syntax/train/annotations/train.json \
    --val_img    arcade/syntax/val/images \
    --val_ann    arcade/syntax/val/annotations/val.json \
    --input_size 256 \
    --epochs     100 \
    --batch_size 16 \
    --device     cuda

---
## Download Checkpoints

In [ ]:
# ── 10. Download CMT best checkpoint ─────────────────────────────────────
from google.colab import files
files.download('checkpoints/best.pth')

In [ ]:
# ── 11. Download pretrained backbone (optional) ───────────────────────────
from google.colab import files
import glob
best = glob.glob('checkpoints/*_best.pth.tar')
if best:
    files.download(best[0])

---
## Nach dem Download — lokal einrichten

```bash
# Backbone
mv ~/Downloads/resnet50_best.pth.tar checkpoints/
ln -sf checkpoints/resnet50_best.pth.tar place2.pth

# CMT Checkpoint
mv ~/Downloads/best.pth checkpoints/
```

Inference lokal:
```bash
python demo.py \
  --ckpt checkpoints/best.pth \
  --img_path ./samples/test_img \
  --mask_path ./samples/test_mask \
  --output_path ./samples/results \
  --device cpu
```